# CAPÍTULO 03 - Estrutura de dados embutidas, funções e arquivos

### Nota Técnica [83]

In [1]:
# imports se necessário
import itertools
import re

## Funções

São o principal e mais importante método de organização de código e reutilização em Python. Também podem ajudar você a deixar seu código mais legível ao dar um nome a um grupo de instruções Python.

As funções são declaradas com a palavra reservada `def`, e o retorno é feito com a palavra reservada `return`:

```python
def my_functioon(x, y, z=1.5):
    if z > 1:
        return z * (x + y)
    else:
        return z / (x + y)
```

Não há probelmas em ter várias instruções `return`. Se Python alcançar o final de uma função sem encontrar um instrução `return`, `None` será devolvido automaticamente.

Toda função pode ter argumentos _posicionais_ e argumentos _nomeados_. Argumentos nomeados são mais comumente utilizados para especificar valores default ou argumentos opcionais. Na função anterior, x e y são argumentos posicionais,enquanto z é um argumento nomeado.

Iss significa que a função pode ser chamada de qualquer uma destas maneiras:

```python
my_function(5, 6, z=0.7)
my_function(3.14, 7, 3.5)
my_function(10, 20)
```

a principal restrição para os arumentos de função é que argumentos nomeados _devem_ vir depois dos argumentos posicionais (se houver). 

### Namespaces, escopo e funções locais

As funções podem acessar variáveis em dois escopos diferentes: _global_ e _local_. Um nme alternativo e mais descritivo para o escopo e uma variável em Python é _namespace_. Quaisquer variáveis que recebam uma atribuição em uma função, por padrão são associados ao namespace local, criado quando a função é chamada e imediatamente preenchido com argumentos da função. epois que a função termina, o namespace local é destruído (com algumas exceções). Considere a função:

```python
def func():
    a = []
    for i in range(5):
        a.append(i)
```

Quando `func()` é chamada, a lista vazia `a` é criada, cinco elementos são concatenados e então `a` é destruída quando a função termina. Suponha que ao invés tivessemos declarado assim:

```python
a = []
def func():
    for i in range(5):
        a.append(i)
```

Fazer uma atribuição a uma variável fora do escopo da função é possível, porém essas variáveis devem ser declaradas como globais com a apalavra reservada `global`:

In [2]:
a = None


def bind_a_variable():
    global a
    a = []


bind_a_variable()

print(a)

[]


<div style="border:2px solid #ffc107; padding:10px; background-color:#fff3cd; border-radius:5px;">
⚠️ <b>Aviso:</b> Geralmente não é incentivado o uso de <i>global</i>. As variáveis globais são comumente usadas para armazenar algum tipo de estado em um sistema. Talvez seja melhor utilizar POO (classes)
</div>

### Devolvendo diversos valores

Exemplo:

```python
def f():
    a = 5
    b = 6
    c = 7
    return a, b, c

a, b, c = f(c)
```

Em análise de dados e em outras aplicaçõescientíficas, voce poderá se ver fazendo isso com frequência. O que está acontecendo neste exemplo é que a função, na verdade, devolve apenas _um_ objeto, uma tupla no caso, que está sendo desempacotado nas variáveis de resultado. No exemplo anterior, poderíamos ter feit o seguinte, como alternativa:

```python
return_value = f()
```

Nesse caso, `return_value` seria uma tupla de três, com as três variáveis devolvidas. Uma alternativa possivelmente atraente a devolver diversos valores como fizemos antes pode ser devolver um dicionário em seu lugar:

```python
def f():
    a = 5
    b = 6
    c = 7
    return {'a': a, 'b': b, 'c': c}
```

### Funções são objetos

Como funções Python são objetos, muitas construções podem ser facilmente expressas, as quais seriam difíceis em outras linguagens. Suponha que estejamos fazendo uma limpeza de dados e precisemos aplicar uma série de transformações na lista de strings a seguir:

In [3]:
states = [
    "SANTA catarina",
    "PArAna!",
    "Rio grande do sul",
    "São Paulo",
    "RIO de Janeiro#",
]

Qualquer pessoa que já tenha trabalhado alguma vez com dados de pesquisa submetidos por usuários já viu resultados desorganizados como esses. Muitas tarefas precisam ser feitas para deixar essa lista de strings uniforme e pronta para a análise. Uma forma é usar métodos embutidos de string, junto com o módulo `re` da bibliteca-padrão para expressões regulares:

In [4]:
def clean_strings(strings):
    result = []
    for value in strings:
        value = value.strip()
        value = re.sub("[!#]", "", value)
        value = value.title()
        result.append(value)
    return result


clean_strings(states)

['Santa Catarina',
 'Parana',
 'Rio Grande Do Sul',
 'São Paulo',
 'Rio De Janeiro']

Uma abordagem alternativa, que talvez lhe pareça conveniente, é criar uma lista das operações que você deseja aplicar em um conjunto particular de strings:

In [5]:
def remove_punctuation(value):
    return re.sub("[!?#$]", "", value)

In [6]:
clean_ops = [str.strip, remove_punctuation, str.title]

In [7]:
def clean_strings(strings, ops):
    result = []
    for value in strings:
        for function in ops:
            value = function(value)
        result.append(value)
    return result

In [8]:
clean_strings(states, clean_ops)

['Santa Catarina',
 'Parana',
 'Rio Grande Do Sul',
 'São Paulo',
 'Rio De Janeiro']

Um padrão mais _funcional_ como esse permite modificar facilmente o modo como as strings são transformadas em um nível bem alto. A função `clean_strings` agora também está mais reutilizável e genérica.

Podemos usar funções como argumentos para outras funções, como a função embutida `map`, que aplica uma função a uma sequência de algum tipo:

In [9]:
for x in map(remove_punctuation, states):
    print(x)

SANTA catarina
PArAna
Rio grande do sul
São Paulo
RIO de Janeiro


### Funções anônimas (lambdas)

Python oferece suporte para o que chamamos de funçõesfunçẽs _anônimas_ ou _lambdas_ -- uma forma de escrever funções constituídas de uma única instrução, cujo resultado é o valor de retorno. Elas são definidas com a palavra reservada _lambda_, que não tem outro significado a não ser dizer que "estamos declarando uma função aônima":

```python
def short_function(x):
    return x * 2

equiv_anon = lambda x: x * 2
```

Funções lambdas são particularmente conveninetes na análise de dados, pois, como veremos, há muitos casos em que as funções de transformação de dados aceitarão funções como argumentos. O que normalmente exige-se menos digitação  passar uma função lambda, em oposição a escrever uma declaração de função completa ou até mesmo atribuir a função lambda a uma variável local.

```python
def apply_to_list(some_list, f):
    return [f(x) for x in some_list]

ints = [4, 0, 1, 5, 6]
apply_to_list(ints, lambda x: x * 2)
```

Neste caso daria para simplismente fazer `[x * 2 for x in ints]`, mas, nesse caso, pudemos passar suncitamente um operador personalizado para a função `apply_to_list`. Neste outro exemplo, queremos ordenar uma coleção de _strings_ de acordo com o número de letras distintas em cada string:

In [10]:
strings = ["foo", "card", "bar", "aaaa", "abab"]

strings.sort(key=lambda x: len(set(list(x))))
strings

['aaaa', 'foo', 'abab', 'bar', 'card']

<div style="border:2px solid #17a2b8; padding:10px; background-color:#d1ecf1; border-radius:5px;">
🔎 <b>Observação:</b> um dos motivos para a função lambda serem chamadas de funções anônimas é que, de modo diferente das funções declaradas com a a palavra reservada <b>def</b>, o objeto função proprieamente dito jamais recebe um atributo <b>__name__</b> explícito.
</div>

### Currying: aplicação parcial dos argumentos

_Currying_ faz parte do jargão da ciência da computação e significa derivar novas funções a partir de funções existentes por meio de _aplicação parcial de argumentos_. Por exemplo, suponha que tenhamos uma função trivial que some dois números:

```python
def add_numbers(x, y):
    return x + y
```

Usando essa função, poderíamos derivar uma nova função de uma variável , `add_five`, que some 5 ao seu argumento:

```python
add_five = lambda y: add_numbers(5, y)
```

Dizemos que o segundo argumento de `add_numbers` sofreu `currying`. não há nada muito sofisticado nesse caso, pois tudo que realmente fizemos foi definir uma nova função que chama uma função existente. O módulo embutido `functools` pode simplificar esse processo usando a função `partial`:

```python
from functools import partial

add_five = partial(add_numbers, 5)
```

### Geradores

Ter uma forma consistente de iterar por sequências, como objetos em uma lista ou linhas em um arquivo, é um recurso importante de Python. Isso é feito por meio do _protocolo iterador_ -- um modo genérico de deixar os objetos iteráveis. Por exemplo, iterar por um dicionário produz as suas chaves:

In [11]:
some_dict = {"a": 1, "b": 2, "c": 3}

for key in some_dict:
    print(key)

a
b
c


Quando escrevemos `for key in some_dict`, o interpretador Python inicialmente tenta criar um iterador a partir de `some_dict`:

In [12]:
dict_iterator = iter(some_dict)
dict_iterator

Um iterador é qualquer objeto que produzirá objetos ao interpretador Python quando usado em um contexto como um laço `for`. A maioria dos métodos que espera uma lista ou um objeto do tipo lista também aceitará qualquer objeto iterável. Isso inclui métodos embutidos como `min`, `max` e `sum`, além de construtores de tipo como `list` e `tuple`

In [13]:
list(dict_iterator)

['a', 'b', 'c']

Um _gerador_ é uma forma concisa de construir um novo objeto iterável. Enquanto funções usuais executam e devolvem um único resultado de cada vez, os geradores devolvem um sequência de vários resultados em modo lazy, fazendo uma pausa após cada um, até que o próximo resultado seja solicitado. Para criar um gerador, utilize a palavra reservada `yield` em vez de `return` em uma função:

```python
def squares(n=10):
    print('Generating squares from 1 to {0}'.format(n ** 2))
    for i in range(1, n + 1):
        yield i ** 2
```

Quando você realmente chamar o gerador, nenhum código será executado imediatamente:

In [14]:
def squares(n=10):
    print("Gerando raiz quadrada de 1 a {0}".format(n**2))
    for i in range(1, n + 1):
        yield i**2


gen = squares()
gen

<generator object squares at 0x7094ebd4f680>

O gerador só passará a executar o seu código quando você começar a lhe solicitar elementos:

In [15]:
for x in gen:
    print(x, end=" ")

Gerando raiz quadrada de 1 a 100
1 4 9 16 25 36 49 64 81 100 

Outra forma mais concisa ainda de criar um gerador é usando uma _expressão geradora_. É um gerador análogo a list, dict e set comprehensions; para criar um, coloque o que seria uma list comprehension entre parênteses em vez de usar colchetes:

In [16]:
gen = (x**2 for x in range(100))
gen

<generator object <genexpr> at 0x7095194b85f0>

Isso é totalmente equivalente ao gerador seguinte, mais extenso:

```python
def _make_gen():
    for x in range(100):
        yield x ** 2

gen - _make_gen()
```

Expressões geradoras podem ser usadas no lugar de list comprehensions como argumentos de função em muitos casos:

In [17]:
sum(x**2 for x in range(100))

328350

In [18]:
dict((i, i**2) for i in range(5))

{0: 0, 1: 1, 2: 4, 3: 9, 4: 16}

### Módulo itertools

O módulo itertools da biblioteca-padrão tem uma coleção de geradores para muitos algoritmos comuns de dados. Um exemplo é `groupby`que aceita qualquer sequência e uma função, agrupando elementos consecutivos da sequência de acordo com o vaor de retorno da função:

In [19]:
first_letter = lambda x: x[0]

names = [
    "Bruno",
    "Leonardo",
    "Leanro",
    "Túlio",
    "Lucas",
    "Bianca",
    "Marcelo",
    "José",
    "Maria",
]

for letter, names in itertools.groupby(names, first_letter):
    print(letter, list(names))

B ['Bruno']
L ['Leonardo', 'Leanro']
T ['Túlio']
L ['Lucas']
B ['Bianca']
M ['Marcelo']
J ['José']
M ['Maria']


**Algumas funções úteis de itertools**

| Função | Descrição |
|--------|-----------|
| combinations(iterable, k) | Gera uma sequência de todas as tuplas possíveis de k elementos no iterável, ignorando a ordem e sem substituição |
| permutations(iterable, k) | Gera uma sequência de todas as tuplas possíveis de k elementos no iterável, respeitando a ordem |
| groupby(iterable[, keyfunc]) | Gera (key, sub-iterator) para cada chave única |
| product(*iterables, repeat=1) | Gera o produto cartesiano dos iteráveis de entrada com tuplas, de modo semelhante a um laço `for` aninhado |

### Erros e tratamento de exceção

Tratar erros ou _exceções_ em Python e modo elegante é uma parte importante na construção de programas robustos. Em aplicações de análise de dados, muitas funções servem somente para determinados tipos de entrada. Como exemplo, a função `float` de Python é capaz de fazer cast de uma string para um número de ponto flutuante, porém falha com `ValueError` se os dados de entrada forem inadequados:

```bash
In: float('1.2345')
Out: 1.2345

In: float('something')
-----------------------------
Value Error                 Traceback (most recent call last)
<ipython-input-198-2649e4ade0e6> in <module>()
----> 1 float('something')
Value Error: could not convert string to float: 'something'
```

Suponha que quiséssemos uma versão de `float` que falhasse de modo elegante, devolvendo o argumento de entrada. Podemos fazer isso escrevendo uma função que encapsule a chamada a `´float` em um bloco `try/catch`:

```python
def attempt_float(x):
    try:
        return float(x)
    except:
        return x
```

O código na parte `except` do bloco será executado somente se `float(x)` gerar uma exceção; `float` é capaz de gerar outras exceções além de `ValueError`, como por exemplo `TypeError`; Talvez você queira suprimir apenas `ValueError`, pois `TypeError` pode indicar um _bug_ legítimo em seu programa.

```python
def attempt_float(x):
    try:
        return float(x)
    except ValueError:
        return x
```

Ou se preferir capturar vários tipos de exceção escrevendo uma tupla de tipos de exceção:

```python
def attempt_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return x
```

Agora se você não quiser sumprimir uma exceção, mas sim que algum código seja executado, independentemente de o código no bloc `try` ter sido bem-sucedido ou não, utilizamos o `finally`:

```python
f = open(path, 'w')

try:
    write_to_file(f)
finally:
    f.close()
```

Neste caso, o handle do arquivo f _sempre será fechado_. Também podemos ter um código que execute somente se o bloc `try` for bem-sucedido usando `else`:

```python
f = open(path, 'w')

try:
    write_to_file(f)
except:
    print('Failed')
else:
    print('Succeeded')
finally:
    f.close()
```

## Parecer Técnico GPT 5.2

Parecer técnico

1. Execução e reprodutibilidade (20%) — 90/100
   O notebook executa do início ao fim sem erro (imports mínimos, exemplos determinísticos). Ponto de atenção: não há célula explícita de “ambiente/requisitos”, mas isso é aceitável para bloco didático.

2. Correção técnica do código e lógica (25%) — 82/100
   Em geral correto e alinhado ao tema (funções, lambdas, iteradores/geradores, itertools). Ajustes recomendados:

* Na função `squares`, o print diz “raiz quadrada”, mas o gerador produz quadrados (mensagem inconsistente com a lógica).
* `itertools.groupby` agrupa por blocos consecutivos; para “agrupar de verdade por chave”, a lista precisa estar ordenada pela mesma chave (isso não está explicitado, e a lista `names` não está ordenada).
* Há redefinição de `clean_strings` (segunda definição sobrescreve a primeira). Se isso for intencional (refatoração didática), vale marcar explicitamente no texto para não parecer “erro”.

3. Organização e legibilidade do notebook (15%) — 88/100
   Boa segmentação por seções, alternância saudável entre markdown e código, exemplos pequenos. Pequena perda por repetição de função com o mesmo nome sem aviso.

4. Texto/PT-BR e comunicação (10%) — 85/100
   Texto bem conduzido. Pequenas inconsistências/ruídos: a mensagem “raiz quadrada” vs quadrados; e faltam observações conceituais em pontos que costumam confundir (ex.: exaustão de iteradores, `groupby` exigir ordenação).

5. Análise e interpretação (15%) — 75/100
   O notebook demonstra, mas interpreta pouco. O que elevaria muito este critério:

* Explicar o “iterator exhaustion” (por que após `list(dict_iterator)` ele se esgota).
* Explicar o comportamento de `groupby` com dados não ordenados e mostrar a correção (`sorted(names, key=first_letter)`).

6. Boas práticas e higiene (10%) — 78/100
   Como material didático, ok. Mas:

* Exemplo com `global a` é um antipadrão; se for para ensino, vale rotular como “evitar em produção” e sugerir alternativa (retornar valor; encapsular em classe; closure).
* Regex de pontuação difere entre exemplos (`[!#]` vs `[!?#$]`); se intencional, explique o motivo.

7. Profundidade (5%) — 72/100
   Cobertura correta, mas ainda “expositiva”. Um ou dois mini-exercícios/guias (ex.: “transforme X usando ops”, “corrija groupby”, “reescreva com generator expression”) aumentariam a profundidade.

Nota Técnica (B1): 83/100.